In [1]:
import pandas as pd
from google.cloud import bigquery
from google.oauth2 import service_account

credentials = service_account.Credentials.from_service_account_file("key.json") # lokalizacja pobranego klucza z punktu 1.4.

client = bigquery.Client(credentials=credentials, project=credentials.project_id) 

In [2]:
gsod:pd.DataFrame
if False:
    query = """
    SELECT *
    FROM `bigquery-public-data.noaa_gsod.gsod2020`
    """

    query_job = client.query(query)
    query_result = query_job.result()
    gsod = query_result.to_dataframe()
    gsod.to_csv("gsod2020.csv.gz", index=False)
else:
    gsod = pd.read_csv("gsod2020.csv.gz")


/tmp/ipykernel_3540169/3958620364.py:13: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  gsod = pd.read_csv("gsod2020.csv.gz")


In [7]:
gsod

,stn,wban,date,year,mo,da,temp,count_temp,dewp,count_dewp,...,flag_min,prcp,flag_prcp,sndp,fog,rain_drizzle,snow_ice_pellets,hail,thunder,tornado_funnel_cloud
0,013590,99999,2020-09-27,2020,9,27,42.4,4,41.3,4,...,*,0.00,G,999.9,0,0,0,0,0,0
1,024960,99999,2020-09-01,2020,9,1,63.7,4,9999.9,0,...,NaN,0.00,G,999.9,0,0,0,0,0,0
2,039610,99999,2020-09-03,2020,9,3,61.9,4,51.7,4,...,*,0.03,A,999.9,0,1,0,0,0,0
3,085020,99999,2020-08-26,2020,8,26,73.9,4,69.4,4,...,*,0.00,I,999.9,0,0,0,0,0,0
4,085020,99999,2020-09-11,2020,9,11,73.3,4,67.7,4,...,*,0.00,I,999.9,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4119200,A07141,327,2020-08-28,2020,8,28,66.3,24,63.2,24,...,*,99.99,NaN,999.9,0,1,0,0,0,0
4119201,A07354,132,2020-07-21,2020,7,21,66.7,24,60.2,24,...,*,0.00,G,999.9,0,1,0,0,0,0
4119202,A07359,240,2020-08-18,2020,8,18,66.2,24,45.8,24,...,*,0.00,I,999.9,0,0,0,0,0,0
4119203,A51256,451,2020-03-09,2020,3,9,56.3,24,42.9,24,...,*,0.00,I,999.9,0,0,0,0,0,0


In [14]:
gsod.describe().T

,count,mean,std,min,25%,50%,75%,max
year,4119205.0,2020.000000,0.000000,2020.0,2020.0,2020.0,2020.00,2020.00
mo,4119205.0,6.487342,3.463681,1.0,3.0,6.0,10.00,12.00
da,4119205.0,15.748153,8.809447,1.0,8.0,16.0,23.00,31.00
temp,4119205.0,55.551540,22.468503,-105.0,41.7,57.8,73.30,110.00
count_temp,4119205.0,18.173296,7.416018,4.0,8.0,24.0,24.00,24.00
dewp,4119205.0,533.365322,2151.760493,-112.6,32.1,47.0,62.60,9999.90
count_dewp,4119205.0,17.080642,8.238555,0.0,8.0,23.0,24.00,24.00
slp,4119205.0,4097.741160,4265.959734,933.5,1011.9,1020.1,9999.90,9999.90
count_slp,4119205.0,10.347779,9.762281,0.0,0.0,8.0,23.00,24.00
stp,4119205.0,699.783492,432.008626,0.0,20.3,972.0,999.90,999.90


In [8]:
isd = pd.read_csv("isd-history.csv")
isd.head()

,USAF,WBAN,STATION NAME,CTRY,STATE,ICAO,LAT,LON,ELEV(M),BEGIN,END
0,007018,99999,WXPOD 7018,NaN,NaN,NaN,0.00,0.000,7018.0,20110309,20130730
1,007026,99999,WXPOD 7026,AF,NaN,NaN,0.00,0.000,7026.0,20120713,20170822
2,007070,99999,WXPOD 7070,AF,NaN,NaN,0.00,0.000,7070.0,20140923,20150926
3,008260,99999,WXPOD8270,NaN,NaN,NaN,0.00,0.000,0.0,20050101,20120731
4,008268,99999,WXPOD8278,AF,NaN,NaN,32.95,65.567,1156.7,20100519,20120323


In [10]:
isd["WBAN"] = isd["WBAN"].apply(lambda x: str(x).zfill(5))
gsod["wban"] = gsod["wban"].apply(lambda x: str(x).zfill(5))
df = gsod.merge(isd, how="left", left_on=["stn", "wban"], right_on=["USAF", "WBAN"])

3.1 Ilość wierszy z danymi

In [11]:
len(df)

4119205

Ilość krajów

In [12]:
len(df["CTRY"].unique())

263

In [13]:
df[["stn", "wban", "year", "mo", "da"]].head()

,stn,wban,year,mo,da
0,013590,99999,2020,9,27
1,024960,99999,2020,9,1
2,039610,99999,2020,9,3
3,085020,99999,2020,8,26
4,085020,99999,2020,9,11


Jak zapisane są wartości liczbowe

In [15]:
df["temp"]

0          42.4
1          63.7
2          61.9
3          73.9
4          73.3
           ... 
4119200    66.3
4119201    66.7
4119202    66.2
4119203    56.3
4119204    65.5
Name: temp, Length: 4119205, dtype: float64